# Merges y Joins en Pandas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/13_pandas_io/code/02_merges_y_joins.ipynb)

Combinar tablas es una de las operaciones más comunes en análisis de datos. En este notebook vas a aprender a usar `pd.merge()` — la función central para joins en pandas — y a entender qué pasa exactamente con tus filas en cada tipo de join.

Al terminar vas a poder:
- Hacer inner, left, right y outer joins
- Manejar columnas con nombres distintos
- Hacer merge por índice
- Detectar y evitar el problema de la explosión de filas
- Elegir entre `pd.merge()` y `df.join()`

In [1]:
%pip install pandas==2.2.3 numpy==1.26.4

In [1]:
!pip install --upgrade numpy pandas

In [2]:
import pandas as pd
import numpy as np

## Datasets que usaremos a lo largo del notebook

Vamos a usar tres tablas pequeñas y relacionadas: clientes, pedidos y productos. Así puedes ver exactamente qué filas entran y salen en cada operación.

In [3]:
# Tabla de clientes — no todos han hecho pedidos
clientes = pd.DataFrame({
    'id_cliente': [1, 2, 3, 4, 5, 6],
    'nombre':     ['Ana', 'Bruno', 'Carla', 'Diego', 'Elena', 'Felipe'],
    'ciudad':     ['CDMX', 'GDL', 'MTY', 'CDMX', 'GDL', 'MTY'],
})

# Tabla de pedidos — solo algunos clientes han comprado
pedidos = pd.DataFrame({
    'id_pedido':  [101, 102, 103, 104, 105],
    'id_cliente': [1,   2,   1,   3,   7],   # 7 NO existe en clientes
    'id_producto':[10,  11,  12,  10,  11],
    'cantidad':   [2,   1,   3,   1,   2],
})

# Tabla de productos
productos = pd.DataFrame({
    'id_producto': [10, 11, 12, 13],
    'nombre_prod': ['Laptop', 'Mouse', 'Teclado', 'Monitor'],
    'precio':      [15000, 350, 800, 5500],
})

print("=== clientes ===")
print(clientes)
print("\n=== pedidos ===")
print(pedidos)
print("\n=== productos ===")
print(productos)

=== clientes ===
   id_cliente  nombre ciudad
0           1     Ana   CDMX
1           2   Bruno    GDL
2           3   Carla    MTY
3           4   Diego   CDMX
4           5   Elena    GDL
5           6  Felipe    MTY

=== pedidos ===
   id_pedido  id_cliente  id_producto  cantidad
0        101           1           10         2
1        102           2           11         1
2        103           1           12         3
3        104           3           10         1
4        105           7           11         2

=== productos ===
   id_producto nombre_prod  precio
0           10      Laptop   15000
1           11       Mouse     350
2           12     Teclado     800
3           13     Monitor    5500


---
## Sección 1: El concepto — merge vs SQL JOIN

Si alguna vez usaste SQL, ya conoces los JOINs. En pandas la función equivalente es `pd.merge()`. El concepto es el mismo: combinar dos tablas usando una columna (o varias) como llave.

```
INNER JOIN — solo los que coinciden en ambas tablas
  A: [1, 2, 3]    B: [2, 3, 4]
  resultado: [2, 3]

LEFT JOIN — todos los de A, con lo que haya en B (NaN si no hay)
  A: [1, 2, 3]    B: [2, 3, 4]
  resultado: [1, 2, 3]  (1 tiene NaN en columnas de B)

RIGHT JOIN — todos los de B, con lo que haya en A (NaN si no hay)
  A: [1, 2, 3]    B: [2, 3, 4]
  resultado: [2, 3, 4]  (4 tiene NaN en columnas de A)

OUTER JOIN — todos de A y B, NaN donde no hay match
  A: [1, 2, 3]    B: [2, 3, 4]
  resultado: [1, 2, 3, 4]  (1 y 4 tienen NaN en columnas del otro)
```

**Diferencia clave entre pandas:**
- `pd.merge()` — función principal, merge por columnas o índices, muy flexible
- `df.join()` — atajo para merge por índice, más cómodo en ciertos casos

---
## Sección 2: Inner join — solo los que coinciden

El inner join es el tipo de merge por defecto. Solo conserva las filas que tienen match en **ambas** tablas. Si un cliente no aparece en pedidos, desaparece del resultado.

In [4]:
# how='inner' es el valor por defecto, pero lo ponemos explícito
inner = pd.merge(clientes, pedidos, on='id_cliente', how='inner')
print(inner)

   id_cliente nombre ciudad  id_pedido  id_producto  cantidad
0           1    Ana   CDMX        101           10         2
1           1    Ana   CDMX        103           12         3
2           2  Bruno    GDL        102           11         1
3           3  Carla    MTY        104           10         1


Veamos exactamente qué filas se perdieron y por qué.

In [5]:
# Clientes sin pedidos — no aparecen en el inner join
clientes_en_pedidos = pedidos['id_cliente'].unique()
perdidos_de_clientes = clientes[~clientes['id_cliente'].isin(clientes_en_pedidos)]
print("Clientes que NO tienen pedidos (se pierden en inner join):")
print(perdidos_de_clientes)

print()

# Pedidos con id_cliente que no existe en clientes
ids_clientes = clientes['id_cliente'].unique()
perdidos_de_pedidos = pedidos[~pedidos['id_cliente'].isin(ids_clientes)]
print("Pedidos con id_cliente inexistente (también se pierden en inner join):")
print(perdidos_de_pedidos)

Clientes que NO tienen pedidos (se pierden en inner join):
   id_cliente  nombre ciudad
3           4   Diego   CDMX
4           5   Elena    GDL
5           6  Felipe    MTY

Pedidos con id_cliente inexistente (también se pierden en inner join):
   id_pedido  id_cliente  id_producto  cantidad
4        105           7           11         2


**Resumen:** inner join solo conserva las intersección. Clientes 4, 5, 6 no hicieron pedidos y desaparecen. El pedido 105 del cliente 7 (que no existe en clientes) también desaparece.

💡 **Prueba esto:** Cambia `how='inner'` por `how='left'` en la celda de arriba. ¿Cuántas filas aparecen ahora?

### Ejercicio 2

Haz un inner join entre `pedidos` y `productos` usando `id_producto` como llave. ¿Cuántas filas tiene el resultado? ¿Qué producto no aparece y por qué?

In [6]:
res = pd.merge(pedidos, productos, on='id_producto', how = 'inner')
print(res.head())


   id_pedido  id_cliente  id_producto  cantidad nombre_prod  precio
0        101           1           10         2      Laptop   15000
1        102           2           11         1       Mouse     350
2        103           1           12         3     Teclado     800
3        104           3           10         1      Laptop   15000
4        105           7           11         2       Mouse     350


---
## Sección 3: Left join — todos los de la izquierda

El left join conserva **todas** las filas del DataFrame izquierdo. Si no hay match en el derecho, las columnas de la derecha se rellenan con `NaN`.

Caso de uso típico: "quiero todos mis clientes, y si tienen pedidos que aparezcan, pero no quiero perder a los que no compraron".

In [7]:
left = pd.merge(clientes, pedidos, on='id_cliente', how='left')
print(left)

   id_cliente  nombre ciudad  id_pedido  id_producto  cantidad
0           1     Ana   CDMX      101.0         10.0       2.0
1           1     Ana   CDMX      103.0         12.0       3.0
2           2   Bruno    GDL      102.0         11.0       1.0
3           3   Carla    MTY      104.0         10.0       1.0
4           4   Diego   CDMX        NaN          NaN       NaN
5           5   Elena    GDL        NaN          NaN       NaN
6           6  Felipe    MTY        NaN          NaN       NaN


Los clientes 4, 5 y 6 aparecen con `NaN` en las columnas de pedidos — no hicieron ningún pedido, pero siguen en la tabla.

In [8]:
# Filtrar exactamente los clientes sin pedidos
# id_pedido es NaN solo cuando no hubo match en pedidos
sin_pedidos = left[left['id_pedido'].isna()]
print("Clientes sin pedidos:")
print(sin_pedidos[['id_cliente', 'nombre', 'ciudad']])

Clientes sin pedidos:
   id_cliente  nombre ciudad
4           4   Diego   CDMX
5           5   Elena    GDL
6           6  Felipe    MTY


Este patrón — left join + filtrar por `isna()` en columna del lado derecho — es la forma estándar de encontrar filas sin correspondencia.

💡 **Prueba esto:** ¿Cuántos clientes por ciudad no han hecho pedidos? Agrupa `sin_pedidos` por `ciudad` y cuenta.

### Ejercicio 3

Haz un left join de `pedidos` con `productos`. Luego encuentra qué pedidos referencian un `id_producto` que NO existe en la tabla de productos.

In [9]:
left = pd.merge(pedidos, productos, on='id_producto', how='left')
print(left)


   id_pedido  id_cliente  id_producto  cantidad nombre_prod  precio
0        101           1           10         2      Laptop   15000
1        102           2           11         1       Mouse     350
2        103           1           12         3     Teclado     800
3        104           3           10         1      Laptop   15000
4        105           7           11         2       Mouse     350


---
## Sección 4: Right join y Outer join

**Right join:** espejo del left — conserva todo el DataFrame derecho, NaN donde no hay match en el izquierdo.

**Outer join:** la unión completa. Conserva filas de ambos lados, NaN donde no hay match en ningún sentido.

In [10]:
# Right join: todos los pedidos, incluso si el cliente no existe en clientes
right = pd.merge(clientes, pedidos, on='id_cliente', how='right')
print("Right join:")
print(right)

Right join:
   id_cliente nombre ciudad  id_pedido  id_producto  cantidad
0           1    Ana   CDMX        101           10         2
1           2  Bruno    GDL        102           11         1
2           1    Ana   CDMX        103           12         3
3           3  Carla    MTY        104           10         1
4           7    NaN    NaN        105           11         2


In [11]:
# Outer join: todo de ambas tablas
outer = pd.merge(clientes, pedidos, on='id_cliente', how='outer')
print("Outer join:")
print(outer)

Outer join:
   id_cliente  nombre ciudad  id_pedido  id_producto  cantidad
0           1     Ana   CDMX      101.0         10.0       2.0
1           1     Ana   CDMX      103.0         12.0       3.0
2           2   Bruno    GDL      102.0         11.0       1.0
3           3   Carla    MTY      104.0         10.0       1.0
4           4   Diego   CDMX        NaN          NaN       NaN
5           5   Elena    GDL        NaN          NaN       NaN
6           6  Felipe    MTY        NaN          NaN       NaN
7           7     NaN    NaN      105.0         11.0       2.0


### El parámetro `indicator=True`

Muy útil para depurar: agrega una columna `_merge` que dice de dónde viene cada fila.

In [12]:
# indicator=True agrega columna _merge con el origen de cada fila
outer_ind = pd.merge(clientes, pedidos, on='id_cliente', how='outer', indicator=True)
print(outer_ind)

   id_cliente  nombre ciudad  id_pedido  id_producto  cantidad      _merge
0           1     Ana   CDMX      101.0         10.0       2.0        both
1           1     Ana   CDMX      103.0         12.0       3.0        both
2           2   Bruno    GDL      102.0         11.0       1.0        both
3           3   Carla    MTY      104.0         10.0       1.0        both
4           4   Diego   CDMX        NaN          NaN       NaN   left_only
5           5   Elena    GDL        NaN          NaN       NaN   left_only
6           6  Felipe    MTY        NaN          NaN       NaN   left_only
7           7     NaN    NaN      105.0         11.0       2.0  right_only


In [13]:
# Ver solo las filas que están en un lado pero no en el otro
print("Solo en clientes (sin pedido):")
print(outer_ind[outer_ind['_merge'] == 'left_only'][['id_cliente', 'nombre', '_merge']])

print("\nSolo en pedidos (cliente inexistente):")
print(outer_ind[outer_ind['_merge'] == 'right_only'][['id_cliente', 'id_pedido', '_merge']])

Solo en clientes (sin pedido):
   id_cliente  nombre     _merge
4           4   Diego  left_only
5           5   Elena  left_only
6           6  Felipe  left_only

Solo en pedidos (cliente inexistente):
   id_cliente  id_pedido      _merge
7           7      105.0  right_only


💡 **Prueba esto:** `indicator=True` funciona con cualquier tipo de join. Úsalo en un inner join — ¿qué valor tiene la columna `_merge` para todas las filas?

### Ejercicio 4

Haz un outer join entre `pedidos` y `productos`. Usa `indicator=True` para encontrar qué productos no tienen ningún pedido asociado.

In [16]:
outer = pd.merge(pedidos, productos, how='outer', indicator=True)
print(outer)

   id_pedido  id_cliente  id_producto  cantidad nombre_prod  precio  \
0      101.0         1.0           10       2.0      Laptop   15000   
1      104.0         3.0           10       1.0      Laptop   15000   
2      102.0         2.0           11       1.0       Mouse     350   
3      105.0         7.0           11       2.0       Mouse     350   
4      103.0         1.0           12       3.0     Teclado     800   
5        NaN         NaN           13       NaN     Monitor    5500   

       _merge  
0        both  
1        both  
2        both  
3        both  
4        both  
5  right_only  


---
## Sección 5: Columnas con nombres distintos

En la vida real las tablas no siempre usan el mismo nombre para la misma columna llave. Para eso existen `left_on` y `right_on`.

In [17]:
# Simulamos que pedidos usa 'client_id' en lugar de 'id_cliente'
pedidos_alt = pedidos.rename(columns={'id_cliente': 'client_id'})
print(pedidos_alt.head(3))

   id_pedido  client_id  id_producto  cantidad
0        101          1           10         2
1        102          2           11         1
2        103          1           12         3


In [18]:
# left_on y right_on cuando las llaves tienen distinto nombre
resultado = pd.merge(
    clientes,
    pedidos_alt,
    left_on='id_cliente',   # columna llave en clientes
    right_on='client_id',   # columna llave en pedidos_alt
    how='inner'
)
print(resultado)

   id_cliente nombre ciudad  id_pedido  client_id  id_producto  cantidad
0           1    Ana   CDMX        101          1           10         2
1           1    Ana   CDMX        103          1           12         3
2           2  Bruno    GDL        102          2           11         1
3           3  Carla    MTY        104          3           10         1


Nota: cuando usas `left_on` y `right_on`, ambas columnas llave aparecen en el resultado (`id_cliente` y `client_id`). Si son redundantes, puedes eliminar una.

In [19]:
# Eliminar la columna redundante después del merge
resultado_limpio = resultado.drop(columns='client_id')
print(resultado_limpio)

   id_cliente nombre ciudad  id_pedido  id_producto  cantidad
0           1    Ana   CDMX        101           10         2
1           1    Ana   CDMX        103           12         3
2           2  Bruno    GDL        102           11         1
3           3  Carla    MTY        104           10         1


### Sufijos cuando hay columnas con el mismo nombre

Si ambas tablas tienen una columna con el mismo nombre (y no es la llave), pandas le agrega sufijos automáticamente.

In [20]:
# Ambas tablas tienen columna 'nombre'
resultado_nombre = pd.merge(clientes, productos, how='cross')  # cross join para el ejemplo
print("Columnas con sufijos automáticos:")
print([c for c in resultado_nombre.columns if 'nombre' in c])

Columnas con sufijos automáticos:
['nombre', 'nombre_prod']


In [22]:
# Controlar los sufijos explícitamente
resultado_sufijos = pd.merge(
    clientes,
    productos,
    how='cross',
    suffixes=('_cliente', '_producto')  # sufijos claros en lugar de _x, _y
)
print("Con sufijos personalizados:")
print([c for c in resultado_sufijos.columns if 'nombre' in c])
print(resultado_sufijos[['nombre', 'nombre_prod']].head(3))

Con sufijos personalizados:
['nombre', 'nombre_prod']
  nombre nombre_prod
0    Ana      Laptop
1    Ana       Mouse
2    Ana     Teclado


💡 **Prueba esto:** Si no defines `suffixes`, pandas usa `_x` y `_y` por defecto. ¿En qué situación eso podría causar confusión en tu análisis?

### Ejercicio 5

Crea una versión de `productos` con la columna `id_producto` renombrada a `prod_id`. Haz un inner join entre `pedidos` y esa tabla usando `left_on` y `right_on`. Luego elimina la columna redundante del resultado.

In [25]:
prod_renombrada = productos.rename(columns={'id_producto': 'prod_id'})
join = pd.merge(pedidos, prod_renombrada, left_on='id_producto', right_on = 'prod_id', how = 'inner')
print(join)



   id_pedido  id_cliente  id_producto  cantidad  prod_id nombre_prod  precio
0        101           1           10         2       10      Laptop   15000
1        102           2           11         1       11       Mouse     350
2        103           1           12         3       12     Teclado     800
3        104           3           10         1       10      Laptop   15000
4        105           7           11         2       11       Mouse     350


---
## Sección 6: Merge por índice

Cuando el índice de un DataFrame es significativo (por ejemplo, el id del cliente), puedes hacer merge directamente por índice en lugar de por columna.

In [26]:
# Poner id_cliente como índice en clientes
clientes_idx = clientes.set_index('id_cliente')
print(clientes_idx)

            nombre ciudad
id_cliente               
1              Ana   CDMX
2            Bruno    GDL
3            Carla    MTY
4            Diego   CDMX
5            Elena    GDL
6           Felipe    MTY


In [27]:
# Merge usando el índice de la izquierda y columna de la derecha
resultado_idx = pd.merge(
    clientes_idx,
    pedidos,
    left_index=True,           # usar el índice de clientes_idx
    right_on='id_cliente',     # usar columna id_cliente de pedidos
    how='inner'
)
print(resultado_idx)

  nombre ciudad  id_pedido  id_cliente  id_producto  cantidad
0    Ana   CDMX        101           1           10         2
2    Ana   CDMX        103           1           12         3
1  Bruno    GDL        102           2           11         1
3  Carla    MTY        104           3           10         1


### `df.join()` — el atajo

Cuando ambos DataFrames tienen el índice como llave, `df.join()` es más conveniente que `pd.merge()`. Internamente es lo mismo.

In [28]:
# Poner id_cliente como índice en pedidos también
pedidos_idx = pedidos.set_index('id_cliente')
print(pedidos_idx)

            id_pedido  id_producto  cantidad
id_cliente                                  
1                 101           10         2
2                 102           11         1
1                 103           12         3
3                 104           10         1
7                 105           11         2


In [29]:
# df.join() es equivalente a pd.merge(left_index=True, right_index=True)
resultado_join = clientes_idx.join(pedidos_idx, how='inner')
print(resultado_join)

           nombre ciudad  id_pedido  id_producto  cantidad
id_cliente                                                
1             Ana   CDMX        101           10         2
1             Ana   CDMX        103           12         3
2           Bruno    GDL        102           11         1
3           Carla    MTY        104           10         1


In [30]:
# La versión equivalente con pd.merge
resultado_merge_eq = pd.merge(
    clientes_idx,
    pedidos_idx,
    left_index=True,
    right_index=True,
    how='inner'
)
# Confirmar que son iguales
print("¿Son iguales?", resultado_join.equals(resultado_merge_eq))

¿Son iguales? True


💡 **Prueba esto:** Usa `clientes_idx.join(pedidos_idx, how='left')`. ¿Qué diferencia ves con respecto al left join que hiciste antes?

### Ejercicio 6

Crea `productos_idx` poniendo `id_producto` como índice de `productos`. Haz un join entre `pedidos_idx` y `productos_idx` usando `left_on='id_producto'` y `right_index=True`.

In [31]:
productos_index = productos.set_index('id_producto')

res = productos_index.merge(pedidos_idx, left_on='id_producto', right_index=True, how='left')
print(res)


            nombre_prod  precio  id_pedido  id_producto  cantidad
id_producto                                                      
10               Laptop   15000        NaN           10       NaN
11                Mouse     350        NaN           11       NaN
12              Teclado     800        NaN           12       NaN
13              Monitor    5500        NaN           13       NaN


---
## Sección 7: Merge por múltiples llaves

A veces una sola columna no es suficiente para identificar una fila de forma única. Puedes pasar una lista de columnas al parámetro `on`.

In [32]:
# Datos de ventas mensuales por tienda
ventas_plan = pd.DataFrame({
    'anio':   [2025, 2025, 2025, 2025, 2025, 2025],
    'mes':    [1,    1,    2,    2,    3,    3   ],
    'tienda': ['A',  'B',  'A',  'B',  'A',  'B' ],
    'meta':   [100,  200,  150,  250,  120,  220 ],
})

ventas_real = pd.DataFrame({
    'anio':   [2025, 2025, 2025, 2025, 2025],
    'mes':    [1,    1,    2,    2,    3   ],
    'tienda': ['A',  'B',  'A',  'B',  'A' ],   # falta B en mes 3
    'ventas': [95,   210,  160,  230,  110 ],
})

print("Plan:")
print(ventas_plan)
print("\nReal:")
print(ventas_real)

Plan:
   anio  mes tienda  meta
0  2025    1      A   100
1  2025    1      B   200
2  2025    2      A   150
3  2025    2      B   250
4  2025    3      A   120
5  2025    3      B   220

Real:
   anio  mes tienda  ventas
0  2025    1      A      95
1  2025    1      B     210
2  2025    2      A     160
3  2025    2      B     230
4  2025    3      A     110


In [33]:
# Merge por combinación de anio + mes + tienda
# Necesitamos las tres para identificar una fila única
comparativo = pd.merge(
    ventas_plan,
    ventas_real,
    on=['anio', 'mes', 'tienda'],  # llave compuesta
    how='left'
)
print(comparativo)

   anio  mes tienda  meta  ventas
0  2025    1      A   100    95.0
1  2025    1      B   200   210.0
2  2025    2      A   150   160.0
3  2025    2      B   250   230.0
4  2025    3      A   120   110.0
5  2025    3      B   220     NaN


In [34]:
# Calcular el cumplimiento de meta
# Si ventas es NaN (sin datos), cumplimiento también será NaN
comparativo['cumplimiento_%'] = (comparativo['ventas'] / comparativo['meta'] * 100).round(1)
print(comparativo)

   anio  mes tienda  meta  ventas  cumplimiento_%
0  2025    1      A   100    95.0            95.0
1  2025    1      B   200   210.0           105.0
2  2025    2      A   150   160.0           106.7
3  2025    2      B   250   230.0            92.0
4  2025    3      A   120   110.0            91.7
5  2025    3      B   220     NaN             NaN


💡 **Prueba esto:** Cambia el `how='left'` a `how='inner'`. ¿Cuántas filas pierdes? ¿Tiene sentido en este contexto de negocio?

### Ejercicio 7

Agrega una tercera tabla `ventas_prev_anio` con las mismas llaves pero para `anio=2024`. Haz un merge entre `ventas_plan` y `ventas_prev_anio` usando `['mes', 'tienda']` como llave (sin incluir `anio`). ¿Qué problema aparece?

In [36]:
ventas_prev_anio = ventas_plan.copy()
ventas_prev_anio['anio'] = 2024

df_comparativo = pd.merge(ventas_plan, ventas_prev_anio, on=['mes', 'tienda'])


---
## Sección 8: Claves duplicadas — el problema silencioso

Este es el error más difícil de detectar en merges. Si hay duplicados en las llaves, pandas hace un **producto cartesiano** entre ellos — y el resultado puede tener muchas más filas de las que esperas, sin ningún error ni advertencia.

In [37]:
# Cliente 1 tiene 2 pedidos (duplicado en pedidos)
pedidos_dup = pd.DataFrame({
    'id_pedido':   [101, 102],
    'id_cliente':  [1,   1  ],   # cliente 1 aparece dos veces
    'id_producto': [10,  11 ],
})

# El producto 10 tiene 2 categorías (duplicado en categorias)
categorias = pd.DataFrame({
    'id_producto': [10,        10      ],  # producto 10 aparece dos veces
    'categoria':   ['Electronico', 'Oferta'],
})

print("pedidos_dup:")
print(pedidos_dup)
print("\ncategorias:")
print(categorias)

pedidos_dup:
   id_pedido  id_cliente  id_producto
0        101           1           10
1        102           1           11

categorias:
   id_producto    categoria
0           10  Electronico
1           10       Oferta


In [38]:
# Merge: pedidos_dup x categorias → explosión de filas
# El pedido 101 (producto 10) se combina con AMBAS categorías de producto 10
explosion = pd.merge(pedidos_dup, categorias, on='id_producto', how='inner')

print(f"Filas en pedidos_dup: {len(pedidos_dup)}")
print(f"Filas en categorias:  {len(categorias)}")
print(f"Filas en resultado:   {len(explosion)}  ← más de las esperadas")
print()
print(explosion)

Filas en pedidos_dup: 2
Filas en categorias:  2
Filas en resultado:   2  ← más de las esperadas

   id_pedido  id_cliente  id_producto    categoria
0        101           1           10  Electronico
1        101           1           10       Oferta


El pedido 101 (producto 10) aparece dos veces porque producto 10 tiene dos categorías. Esto es un **producto cartesiano parcial**.

El peligro: si luego sumas `cantidad * precio`, los valores se cuentan doble. El resultado numérico parece razonable pero está inflado.

In [39]:
# pd.merge con validate para protegerte
# 'one_to_many': la llave debe ser única en el DataFrame IZQUIERDO
try:
    pd.merge(
        pedidos_dup,
        categorias,
        on='id_producto',
        how='inner',
        validate='one_to_one'   # esperamos 1-a-1, pero no es así
    )
except pd.errors.MergeError as e:
    print(f"Error atrapado: {e}")

Error atrapado: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  id_producto
          10 ...


In [40]:
# Opciones de validate:
# 'one_to_one'   → llave única en ambos lados
# 'one_to_many'  → llave única solo en el izquierdo
# 'many_to_one'  → llave única solo en el derecho

# Este merge sí es many_to_one (varios pedidos pueden tener el mismo producto)
# pero categorias tiene duplicados, así que ninguna validación pasará sin limpiar antes
print("Duplicados en categorias por id_producto:")
print(categorias['id_producto'].value_counts())

print("\nDuplicados en pedidos_dup por id_producto:")
print(pedidos_dup['id_producto'].value_counts())

Duplicados en categorias por id_producto:
id_producto
10    2
Name: count, dtype: int64

Duplicados en pedidos_dup por id_producto:
id_producto
10    1
11    1
Name: count, dtype: int64


**Regla práctica:** antes de hacer un merge importante, verifica cuántos duplicados tiene cada tabla en la columna llave con `.value_counts()`. Si esperas una relación 1-a-muchos, usa `validate='one_to_many'` para que te avise si algo está mal.

💡 **Prueba esto:** ¿Qué pasa si tienes duplicados en AMBAS tablas? Crea un ejemplo con 2 filas duplicadas en cada lado y cuenta las filas del resultado.

### Ejercicio 8

Toma `clientes` y crea una versión con el cliente 1 duplicado (dos filas con `id_cliente=1` pero distinta ciudad). Haz un inner join con `pedidos`. Explica por qué el resultado tiene más filas de las esperadas.

In [41]:
cliente_duplicado = clientes.copy()
cliente_duplicado = pd.concat([clientes, cliente_duplicado], ignore_index=True)

res_duplicado = pd.merge(cliente_duplicado, pedidos, on='id_cliente', how = 'inner')

print(res_duplicado)


   id_cliente nombre ciudad  id_pedido  id_producto  cantidad
0           1    Ana   CDMX        101           10         2
1           1    Ana   CDMX        103           12         3
2           2  Bruno    GDL        102           11         1
3           3  Carla    MTY        104           10         1
4           1    Ana   CDMX        101           10         2
5           1    Ana   CDMX        103           12         3
6           2  Bruno    GDL        102           11         1
7           3  Carla    MTY        104           10         1


---
## Sección 9: `merge` vs `join` — cuándo usar cada uno

`df.join()` es simplemente un atajo para `pd.merge()` con `left_index=True`. No hace nada diferente, solo es más corto de escribir en ciertos casos.

| Característica | `pd.merge()` | `df.join()` |
|---|---|---|
| Merge por columna | `on='col'` | No directamente (necesitas `set_index` antes) |
| Merge por índice | `left_index=True, right_index=True` | Por defecto |
| Llaves con distinto nombre | `left_on=`, `right_on=` | `on=` (columna del derecho contra índice del izquierdo) |
| Tipos de join | `how='inner/left/right/outer'` | `how='left/right/inner/outer'` |
| `validate` | Sí | No |
| `indicator` | Sí | No |
| **Cuándo usarlo** | Casi siempre | Cuando el índice ya es la llave y quieres código compacto |

In [42]:
# Los tres son equivalentes cuando los índices son las llaves
ci = clientes.set_index('id_cliente')
pi = pedidos.set_index('id_cliente')

r1 = ci.join(pi, how='left')
r2 = pd.merge(ci, pi, left_index=True, right_index=True, how='left')
r3 = pd.merge(clientes, pedidos, on='id_cliente', how='left').set_index('id_cliente')

print("r1 == r2:", r1.equals(r2))
print("r1 == r3:", r1.sort_index().equals(r3.sort_index()))

r1 == r2: True
r1 == r3: True


---
## Sección 10: Ejercicio integrador

Vamos a construir un reporte completo combinando tres tablas: ventas, productos y tiendas.

In [43]:
# Los tres DataFrames para el ejercicio
ventas = pd.DataFrame({
    'id_venta':    [1, 2, 3, 4, 5, 6, 7],
    'id_producto': [10, 11, 10, 12, 11, 13, 10],
    'id_tienda':   ['T1', 'T1', 'T2', 'T2', 'T3', 'T1', 'T3'],
    'cantidad':    [2, 1, 3, 1, 2, 1, 1],
    'fecha':       pd.to_datetime(['2025-01-05', '2025-01-06', '2025-01-07',
                                   '2025-01-08', '2025-01-09', '2025-01-10', '2025-01-11']),
})

productos_rep = pd.DataFrame({
    'id_producto':  [10, 11, 12, 13],
    'nombre_prod':  ['Laptop', 'Mouse', 'Teclado', 'Monitor'],
    'precio_unit':  [15000, 350, 800, 5500],
    'categoria':    ['Cómputo', 'Periférico', 'Periférico', 'Cómputo'],
})

tiendas = pd.DataFrame({
    'id_tienda': ['T1', 'T2', 'T3', 'T4'],   # T4 no tiene ventas
    'nombre_tienda': ['Norte', 'Sur', 'Centro', 'Oriente'],
    'ciudad': ['CDMX', 'GDL', 'MTY', 'CDMX'],
})

print("ventas:"); print(ventas)
print("\nproductos_rep:"); print(productos_rep)
print("\ntiendas:"); print(tiendas)

ventas:
   id_venta  id_producto id_tienda  cantidad      fecha
0         1           10        T1         2 2025-01-05
1         2           11        T1         1 2025-01-06
2         3           10        T2         3 2025-01-07
3         4           12        T2         1 2025-01-08
4         5           11        T3         2 2025-01-09
5         6           13        T1         1 2025-01-10
6         7           10        T3         1 2025-01-11

productos_rep:
   id_producto nombre_prod  precio_unit   categoria
0           10      Laptop        15000     Cómputo
1           11       Mouse          350  Periférico
2           12     Teclado          800  Periférico
3           13     Monitor         5500     Cómputo

tiendas:
  id_tienda nombre_tienda ciudad
0        T1         Norte   CDMX
1        T2           Sur    GDL
2        T3        Centro    MTY
3        T4       Oriente   CDMX


### Tu tarea:

1. Haz un merge de `ventas` con `productos_rep` para agregar nombre, precio y categoría a cada venta
2. Al resultado anterior, agrega la información de `tiendas` (nombre y ciudad)
3. Calcula la columna `total` = `cantidad * precio_unit`
4. Usa `how='left'` en ambos merges para no perder ventas aunque haya inconsistencias
5. Muestra el reporte final ordenado por `total` de mayor a menor
6. (Bonus) Agrupa por `categoria` y calcula el total vendido por categoría

In [45]:
# Paso 1: ventas + productos_rep
ventas = pd.merge(ventas, productos_rep, on='id_producto', how='left')
print(ventas.head())


   id_venta  id_producto id_tienda  cantidad      fecha nombre_prod_x  \
0         1           10        T1         2 2025-01-05        Laptop   
1         2           11        T1         1 2025-01-06         Mouse   
2         3           10        T2         3 2025-01-07        Laptop   
3         4           12        T2         1 2025-01-08       Teclado   
4         5           11        T3         2 2025-01-09         Mouse   

   precio_unit_x categoria_x nombre_prod_y  precio_unit_y categoria_y  
0          15000     Cómputo        Laptop          15000     Cómputo  
1            350  Periférico         Mouse            350  Periférico  
2          15000     Cómputo        Laptop          15000     Cómputo  
3            800  Periférico       Teclado            800  Periférico  
4            350  Periférico         Mouse            350  Periférico  


In [47]:
# Paso 2: resultado anterior + tiendas
ventas_completo = pd.merge(ventas, tiendas, on='id_tienda', how='left')
print(ventas_completo.columns)


Index(['id_venta', 'id_producto', 'id_tienda', 'cantidad', 'fecha',
       'nombre_prod_x', 'precio_unit_x', 'categoria_x', 'nombre_prod_y',
       'precio_unit_y', 'categoria_y', 'nombre_tienda', 'ciudad'],
      dtype='str')


In [49]:
# Paso 3 y 4: calcular total y ordenar
ventas_completo['total'] = ventas_completo['cantidad'] * ventas_completo['precio_unit_x']

print(ventas_completo[['id_venta', 'cantidad', 'precio_unit_x', 'total']].head())


   id_venta  cantidad  precio_unit_x  total
0         1         2          15000  30000
1         2         1            350    350
2         3         3          15000  45000
3         4         1            800    800
4         5         2            350    700


In [51]:
# Bonus: total por categoría
ventas_por_categoria = ventas_completo.groupby('categoria_x')['total'].sum().reset_index()

ventas_por_categoria = ventas_por_categoria.sort_values(by='total', ascending=False)

print("Ventas totales por categoría:")
print(ventas_por_categoria)


Ventas totales por categoría:
  categoria_x  total
0     Cómputo  95500
1  Periférico   1850


### Solución de referencia

Intenta resolverlo primero. La solución está aquí para que puedas comparar.

In [52]:
# Solución completa del ejercicio integrador

# Paso 1: agregar info de productos
reporte = pd.merge(ventas, productos_rep, on='id_producto', how='left')

# Paso 2: agregar info de tiendas
reporte = pd.merge(reporte, tiendas, on='id_tienda', how='left')

# Paso 3: calcular total
reporte['total'] = reporte['cantidad'] * reporte['precio_unit']

# Ordenar por total descendente
reporte_final = reporte.sort_values('total', ascending=False)

# Mostrar columnas relevantes
cols = ['id_venta', 'fecha', 'nombre_prod', 'categoria', 'nombre_tienda', 'ciudad', 'cantidad', 'precio_unit', 'total']
print(reporte_final[cols].to_string(index=False))

print("\n=== Total por categoría ===")
print(reporte_final.groupby('categoria')['total'].sum().sort_values(ascending=False))

 id_venta      fecha nombre_prod  categoria nombre_tienda ciudad  cantidad  precio_unit  total
        3 2025-01-07      Laptop    Cómputo           Sur    GDL         3        15000  45000
        1 2025-01-05      Laptop    Cómputo         Norte   CDMX         2        15000  30000
        7 2025-01-11      Laptop    Cómputo        Centro    MTY         1        15000  15000
        6 2025-01-10     Monitor    Cómputo         Norte   CDMX         1         5500   5500
        4 2025-01-08     Teclado Periférico           Sur    GDL         1          800    800
        5 2025-01-09       Mouse Periférico        Centro    MTY         2          350    700
        2 2025-01-06       Mouse Periférico         Norte   CDMX         1          350    350

=== Total por categoría ===
categoria
Cómputo       95500
Periférico     1850
Name: total, dtype: int64


---
## Resumen

| Concepto | Qué recordar |
|---|---|
| `pd.merge(df1, df2, on='col', how='inner')` | Solo filas con match en ambos lados |
| `how='left'` | Todas las filas del izquierdo, NaN si no hay match |
| `how='right'` | Todas las filas del derecho, NaN si no hay match |
| `how='outer'` | Todas las filas de ambos lados |
| `indicator=True` | Columna `_merge` para ver el origen de cada fila |
| `left_on=`, `right_on=` | Cuando las llaves tienen distinto nombre |
| `suffixes=('_a', '_b')` | Controlar sufijos en columnas duplicadas |
| `left_index=True` / `right_index=True` | Merge por índice |
| `on=['col1', 'col2']` | Llave compuesta |
| `validate='one_to_many'` | Detectar duplicados inesperados en la llave |
| `df.join(df2)` | Atajo para merge por índice |

**El error más común:** no verificar duplicados en las llaves antes del merge. Siempre revisa con `.value_counts()` o usa `validate=` en merges importantes.